# Cross-Format Covariance Interoperability Tests

Validates COVERX writer (binary + text) and round-trip conversions across
COVERX, COVFIL, and BOXER formats via `CovMat` objects.

In [1]:
import sys, os, tempfile
import numpy as np

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..')))

from kika.cov import read_coverx, write_coverx, read_covfil, write_covfil, read_boxer, write_boxer
from kika.cov.covmat import CovMat
from kika.cov.parse_covmat import _dense_to_banded

# Test file paths
COVERX_BIN  = r'c:\Users\Usuario\BaradDur\Dev\kika\files\cov\scale.rev05.44groupcov'
COVERX_TXT  = r'c:\Users\Usuario\BaradDur\Dev\kika\files\cov\scale.rev05.44groupcov.txt'
COVFIL_MF33 = r'c:\Users\Usuario\BaradDur\Dev\kika\files\cov\260560_80.06.xs.gendf'

tmpdir = tempfile.mkdtemp(prefix='kika_cov_test_')
print(f'Temp dir: {tmpdir}')

Temp dir: C:\Users\Usuario\AppData\Local\Temp\kika_cov_test_z0a0t7xb


## Helper: compare two CovMat objects

In [2]:
def compare_covmats(a: CovMat, b: CovMat, rtol=1e-5, atol=1e-8, label=''):
    """Compare two CovMat objects, reporting per-matrix differences."""
    prefix = f'[{label}] ' if label else ''
    assert a.num_groups == b.num_groups, f'{prefix}num_groups mismatch: {a.num_groups} vs {b.num_groups}'
    assert a.num_matrices == b.num_matrices, f'{prefix}num_matrices mismatch: {a.num_matrices} vs {b.num_matrices}'
    
    # Build lookup for b matrices
    b_lookup = {}
    for i in range(b.num_matrices):
        key = (b.isotope_rows[i], b.reaction_rows[i], b.isotope_cols[i], b.reaction_cols[i])
        b_lookup[key] = b.matrices[i]
    
    max_rel_diff = 0.0
    max_abs_diff = 0.0
    all_pass = True
    
    for i in range(a.num_matrices):
        key = (a.isotope_rows[i], a.reaction_rows[i], a.isotope_cols[i], a.reaction_cols[i])
        assert key in b_lookup, f'{prefix}Matrix {key} not found in second CovMat'
        
        ma = a.matrices[i]
        mb = b_lookup[key]
        
        abs_diff = np.max(np.abs(ma - mb))
        denom = np.max(np.abs(ma))
        rel_diff = abs_diff / denom if denom > 0 else 0.0
        
        max_rel_diff = max(max_rel_diff, rel_diff)
        max_abs_diff = max(max_abs_diff, abs_diff)
        
        if not np.allclose(ma, mb, rtol=rtol, atol=atol):
            print(f'{prefix}FAIL matrix {key}: max_abs={abs_diff:.3e}, max_rel={rel_diff:.3e}')
            all_pass = False
    
    status = 'PASS' if all_pass else 'FAIL'
    print(f'{prefix}{status} -- {a.num_matrices} matrices, max_abs_diff={max_abs_diff:.3e}, max_rel_diff={max_rel_diff:.3e}')
    assert all_pass, f'{prefix}Comparison failed'
    return all_pass

## Test 1: Banded compression unit tests

Verify that `_dense_to_banded` correctly inverts the banded→dense unpacking.

In [3]:
def reconstruct_from_banded(jband, ijj, values, ngroup):
    """Reconstruct dense matrix from banded storage (same as reader logic)."""
    matrix = np.zeros((ngroup, ngroup))
    offset = 0
    for j in range(ngroup):
        if jband[j] > 0:
            start_col = j - ijj[j] + 1
            matrix[j, start_col:start_col + jband[j]] = values[offset:offset + jband[j]]
            offset += jband[j]
    return matrix

# Identity matrix
n = 10
I = np.eye(n)
jb, ij, vals = _dense_to_banded(I)
rec = reconstruct_from_banded(jb, ij, vals, n)
assert np.allclose(I, rec), 'Identity test failed'
print(f'Identity ({n}x{n}): PASS')

# Random symmetric matrix
np.random.seed(42)
A = np.random.randn(n, n)
A = A @ A.T  # symmetric positive semi-definite
jb, ij, vals = _dense_to_banded(A)
rec = reconstruct_from_banded(jb, ij, vals, n)
assert np.allclose(A, rec), 'Random symmetric test failed'
print(f'Random symmetric ({n}x{n}): PASS')

# Sparse banded matrix
B = np.zeros((n, n))
for i in range(n):
    B[i, i] = float(i + 1)
    if i > 0:
        B[i, i-1] = 0.5
    if i < n-1:
        B[i, i+1] = 0.5
jb, ij, vals = _dense_to_banded(B)
rec = reconstruct_from_banded(jb, ij, vals, n)
assert np.allclose(B, rec), 'Sparse banded test failed'
print(f'Sparse banded ({n}x{n}): PASS')

# Zero matrix
Z = np.zeros((n, n))
jb, ij, vals = _dense_to_banded(Z)
rec = reconstruct_from_banded(jb, ij, vals, n)
assert np.allclose(Z, rec), 'Zero matrix test failed'
assert len(vals) == 0, 'Zero matrix should have no values'
print(f'Zero ({n}x{n}): PASS')

print('\nAll banded compression tests passed.')

Identity (10x10): PASS
Random symmetric (10x10): PASS
Sparse banded (10x10): PASS
Zero (10x10): PASS

All banded compression tests passed.


## Test 2: COVERX binary round-trip

Read existing binary → write binary → read back → compare.

In [4]:
cov_bin = read_coverx(COVERX_BIN)
print(f'Read COVERX binary: {cov_bin.num_matrices} matrices, {cov_bin.num_groups} groups')

out_bin = os.path.join(tmpdir, 'roundtrip.coverx')
write_coverx(cov_bin, out_bin, fmt='binary', title='roundtrip test')

cov_bin_rt = read_coverx(out_bin)
print(f'Read back: {cov_bin_rt.num_matrices} matrices, {cov_bin_rt.num_groups} groups')

compare_covmats(cov_bin, cov_bin_rt, rtol=1e-5, atol=1e-8, label='COVERX binary round-trip')

Read COVERX binary: 2524 matrices, 44 groups
Read back: 2524 matrices, 44 groups
[COVERX binary round-trip] PASS -- 2524 matrices, max_abs_diff=0.000e+00, max_rel_diff=0.000e+00


True

## Test 3: COVERX text round-trip

Read existing text → write text → read back → compare.

In [5]:
cov_txt = read_coverx(COVERX_TXT)
print(f'Read COVERX text: {cov_txt.num_matrices} matrices, {cov_txt.num_groups} groups')

out_txt = os.path.join(tmpdir, 'roundtrip.coverx.txt')
write_coverx(cov_txt, out_txt, fmt='text', title='roundtrip text test')

cov_txt_rt = read_coverx(out_txt)
print(f'Read back: {cov_txt_rt.num_matrices} matrices, {cov_txt_rt.num_groups} groups')

compare_covmats(cov_txt, cov_txt_rt, rtol=1e-6, atol=1e-10, label='COVERX text round-trip')

Read COVERX text: 2530 matrices, 44 groups
Read back: 2530 matrices, 44 groups
[COVERX text round-trip] PASS -- 2530 matrices, max_abs_diff=0.000e+00, max_rel_diff=0.000e+00


True

## Test 4: COVFIL → COVERX binary → read back

In [6]:
cov_covfil = read_covfil(COVFIL_MF33)
print(f'Read COVFIL: {cov_covfil.num_matrices} matrices, {cov_covfil.num_groups} groups')

out_cf2bin = os.path.join(tmpdir, 'covfil_to.coverx')
write_coverx(cov_covfil, out_cf2bin, fmt='binary', title='covfil->coverx')

cov_cf2bin = read_coverx(out_cf2bin)
print(f'Read back: {cov_cf2bin.num_matrices} matrices, {cov_cf2bin.num_groups} groups')

compare_covmats(cov_covfil, cov_cf2bin, rtol=1e-5, atol=1e-8, label='COVFIL -> COVERX binary')

Read COVFIL: 7 matrices, 56 groups
Read back: 7 matrices, 56 groups
[COVFIL -> COVERX binary] PASS -- 7 matrices, max_abs_diff=2.655e-08, max_rel_diff=4.093e-08


True

## Test 5: COVFIL → COVERX text → read back

In [7]:
out_cf2txt = os.path.join(tmpdir, 'covfil_to.coverx.txt')
write_coverx(cov_covfil, out_cf2txt, fmt='text', title='covfil->coverx text')

cov_cf2txt = read_coverx(out_cf2txt)
print(f'Read back: {cov_cf2txt.num_matrices} matrices, {cov_cf2txt.num_groups} groups')

compare_covmats(cov_covfil, cov_cf2txt, rtol=1e-6, atol=1e-10, label='COVFIL -> COVERX text')

Read back: 7 matrices, 56 groups
[COVFIL -> COVERX text] PASS -- 7 matrices, max_abs_diff=1.110e-16, max_rel_diff=1.243e-16


True

## Test 6: COVERX → COVFIL → read back

In [8]:
# COVFIL data has standard 5-digit ZAIDs (e.g. 26056) compatible with all formats.
# COVERX-sourced data uses extended 7-digit IDs (e.g. 8001001) that overflow
# COVFIL's ENDF 11-char fields and BOXER's 5-char MAT field.
# So for COVERX -> COVFIL and COVERX -> BOXER, we use COVFIL-sourced data
# that has been round-tripped through COVERX.

cov_for_covfil = read_covfil(COVFIL_MF33)
out_via_cvx = os.path.join(tmpdir, 'via_coverx.coverx')
write_coverx(cov_for_covfil, out_via_cvx, fmt='binary')
cov_via_cvx = read_coverx(out_via_cvx)
print(f'COVFIL -> COVERX: {cov_via_cvx.num_matrices} matrices')

out_cvx2cf = os.path.join(tmpdir, 'coverx_to.covfil')
write_covfil(cov_via_cvx, out_cvx2cf)

cov_cvx2cf = read_covfil(out_cvx2cf)
print(f'COVFIL read back: {cov_cvx2cf.num_matrices} matrices')

# Compare against the COVERX intermediate (float32 precision loss already baked in)
compare_covmats(cov_via_cvx, cov_cvx2cf, rtol=1e-5, atol=1e-8, label='COVERX -> COVFIL')

COVFIL -> COVERX: 7 matrices
COVFIL read back: 7 matrices
[COVERX -> COVFIL] PASS -- 7 matrices, max_abs_diff=4.887e-08, max_rel_diff=2.513e-07


True

## Test 7: COVERX → BOXER → read back

In [9]:
# Re-use COVFIL-sourced data round-tripped through COVERX (compatible ZAIDs)
out_cvx2box = os.path.join(tmpdir, 'coverx_to.boxer')
write_boxer(cov_via_cvx, out_cvx2box, nvf=10)

cov_cvx2box = read_boxer(out_cvx2box)
print(f'Read back: {cov_cvx2box.num_matrices} matrices')

# BOXER uses lower precision (NVF=10 -> 1P8E10.3 -> ~3 decimal digits)
compare_covmats(cov_via_cvx, cov_cvx2box, rtol=1e-2, atol=1e-6, label='COVERX -> BOXER')

Read back: 7 matrices
[COVERX -> BOXER] PASS -- 7 matrices, max_abs_diff=4.428e-05, max_rel_diff=2.277e-04


True

## Test 8: Full chain — COVFIL → COVERX → BOXER → COVFIL

In [10]:
# Start from COVFIL
cov_start = read_covfil(COVFIL_MF33)
print(f'Start (COVFIL): {cov_start.num_matrices} matrices')

# COVFIL -> COVERX binary
step1 = os.path.join(tmpdir, 'chain_step1.coverx')
write_coverx(cov_start, step1, fmt='binary')
cov_step1 = read_coverx(step1)

# COVERX -> BOXER
step2 = os.path.join(tmpdir, 'chain_step2.boxer')
write_boxer(cov_step1, step2, nvf=10)
cov_step2 = read_boxer(step2)

# BOXER -> COVFIL
step3 = os.path.join(tmpdir, 'chain_step3.covfil')
write_covfil(cov_step2, step3)
cov_end = read_covfil(step3)
print(f'End (COVFIL): {cov_end.num_matrices} matrices')

# Cumulative tolerance: float32 + BOXER + COVFIL
compare_covmats(cov_start, cov_end, rtol=1e-2, atol=1e-6, label='Full chain COVFIL->COVERX->BOXER->COVFIL')

Start (COVFIL): 7 matrices
End (COVFIL): 7 matrices
[Full chain COVFIL->COVERX->BOXER->COVFIL] PASS -- 7 matrices, max_abs_diff=4.428e-05, max_rel_diff=2.277e-04


True

## Test 9: CovMat class method API

In [11]:
# Test from_coverx
cov_api = CovMat.from_coverx(COVERX_BIN)
print(f'CovMat.from_coverx(): {cov_api.num_matrices} matrices')

# Use COVFIL-sourced data for API round-trip (compatible ZAIDs)
cov_api_src = read_covfil(COVFIL_MF33)

# Test to_coverx binary
out_api_bin = os.path.join(tmpdir, 'api_test.coverx')
cov_api_src.to_coverx(out_api_bin, fmt='binary', title='API test')
cov_api_rt = CovMat.from_coverx(out_api_bin)
print(f'to_coverx(binary) -> from_coverx: {cov_api_rt.num_matrices} matrices')
compare_covmats(cov_api_src, cov_api_rt, rtol=1e-5, atol=1e-8, label='CovMat API binary')

# Test to_coverx text
out_api_txt = os.path.join(tmpdir, 'api_test.coverx.txt')
cov_api_src.to_coverx(out_api_txt, fmt='text', title='API text test')
cov_api_txt_rt = CovMat.from_coverx(out_api_txt)
print(f'to_coverx(text) -> from_coverx: {cov_api_txt_rt.num_matrices} matrices')
compare_covmats(cov_api_src, cov_api_txt_rt, rtol=1e-5, atol=1e-8, label='CovMat API text')

CovMat.from_coverx(): 2524 matrices
to_coverx(binary) -> from_coverx: 7 matrices
[CovMat API binary] PASS -- 7 matrices, max_abs_diff=2.655e-08, max_rel_diff=4.093e-08
to_coverx(text) -> from_coverx: 7 matrices
[CovMat API text] PASS -- 7 matrices, max_abs_diff=1.110e-16, max_rel_diff=1.243e-16


True

## Test 10: Cross-section preservation through COVERX binary

Verify that `_read_coverx_binary` now extracts cross-sections instead of discarding them.

In [ ]:
import warnings

# 10a: COVFIL cross-sections survive COVERX binary round-trip
cov_src = read_covfil(COVFIL_MF33)
print(f'COVFIL cross-sections: {sorted(cov_src.cross_sections.keys())}')
assert len(cov_src.cross_sections) > 0, 'COVFIL should have cross-sections'

out_xs_rt = os.path.join(tmpdir, 'xs_roundtrip.coverx')
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    write_coverx(cov_src, out_xs_rt, fmt='binary')

cov_xs_rt = read_coverx(out_xs_rt)
print(f'Read-back cross-sections: {sorted(cov_xs_rt.cross_sections.keys())}')
assert len(cov_xs_rt.cross_sections) > 0, 'Cross-sections should survive COVERX binary round-trip'

for key in cov_src.cross_sections:
    assert key in cov_xs_rt.cross_sections, f'Missing cross-section {key}'
    orig = cov_src.cross_sections[key]
    rt = cov_xs_rt.cross_sections[key]
    assert np.allclose(orig, rt, rtol=1e-6), f'XS mismatch for {key}'
    
print('PASS: All cross-sections preserved through COVERX binary round-trip')

# 10b: Synthetic test — explicit values
cm_synth = CovMat(num_groups=3, energy_grid=[1e-5, 0.625, 1e5, 20e6])
cm_synth.add_matrix(1001, 18, 1001, 18, np.eye(3) * 0.01)
cm_synth.cross_sections[(1001, 18)] = np.array([10.5, 20.3, 0.5])
cm_synth.cross_sections[(1001, 102)] = np.array([100.0, 50.0, 1.0])

out_synth = os.path.join(tmpdir, 'xs_synth.coverx')
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    write_coverx(cm_synth, out_synth, fmt='binary')

cm_synth_rt = read_coverx(out_synth)
for key in [(1001, 18), (1001, 102)]:
    assert key in cm_synth_rt.cross_sections, f'Missing XS {key}'
    assert np.allclose(
        cm_synth.cross_sections[key], cm_synth_rt.cross_sections[key], rtol=1e-6
    ), f'XS value mismatch for {key}'

print('PASS: Synthetic cross-section round-trip verified')

## Test 11: MF34 TypeError guards

`write_coverx` and `write_boxer` must reject `MF34CovMat` with a `TypeError`.

In [ ]:
from kika.cov.mf34_covmat import MF34CovMat

mf34 = MF34CovMat()

# write_coverx must reject MF34CovMat
try:
    write_coverx(mf34, os.path.join(tmpdir, 'should_fail.bin'))
    assert False, 'write_coverx should have raised TypeError'
except TypeError as e:
    print(f'write_coverx(MF34CovMat) -> TypeError: {e}')

# write_boxer must reject MF34CovMat
try:
    write_boxer(mf34, os.path.join(tmpdir, 'should_fail.boxer'))
    assert False, 'write_boxer should have raised TypeError'
except TypeError as e:
    print(f'write_boxer(MF34CovMat)  -> TypeError: {e}')

print('PASS: MF34 guards working correctly')

## Test 12: MAT overflow warnings

COVFIL (4-char MAT, max 9999) and BOXER (5-char MAT, max 99999) must warn when
isotope IDs cannot be mapped to valid MAT numbers.

In [ ]:
# Build a CovMat with extended 7-digit ZAIDs (e.g. SCALE IZZZAAA format)
cm_overflow = CovMat(num_groups=2, energy_grid=[1e-5, 1.0, 20e6])
cm_overflow.add_matrix(8001001, 18, 8001001, 18, np.eye(2) * 0.01)

# 12a: COVFIL MAT overflow (4-char, max 9999)
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    write_covfil(cm_overflow, os.path.join(tmpdir, 'overflow_covfil.txt'))
    covfil_warns = [x for x in w if 'COVFIL MAT field' in str(x.message)]
    assert len(covfil_warns) == 1, f'Expected 1 COVFIL MAT warning, got {len(covfil_warns)}'
    print(f'COVFIL MAT overflow: {covfil_warns[0].message}')

# 12b: BOXER MAT overflow (5-char, max 99999)
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    write_boxer(cm_overflow, os.path.join(tmpdir, 'overflow_boxer.txt'))
    boxer_warns = [x for x in w if 'BOXER MAT field' in str(x.message)]
    assert len(boxer_warns) == 1, f'Expected 1 BOXER MAT warning, got {len(boxer_warns)}'
    print(f'BOXER MAT overflow:  {boxer_warns[0].message}')

# 12c: No warning for compatible ZAIDs
cm_compat = CovMat(num_groups=2, energy_grid=[1e-5, 1.0, 20e6])
cm_compat.add_matrix(26056, 18, 26056, 18, np.eye(2) * 0.01)
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    write_covfil(cm_compat, os.path.join(tmpdir, 'no_overflow_covfil.txt'))
    mat_warns = [x for x in w if 'MAT field' in str(x.message)]
    assert len(mat_warns) == 0, f'Unexpected MAT overflow warning for compatible ZAID: {mat_warns}'
    print('No false-positive MAT warning for compatible ZAIDs: PASS')

print('PASS: MAT overflow warnings working correctly')

## Test 13: COVERX format-specific warnings

Verify float32 precision warning (binary) and cross-section loss warning (text).

In [ ]:
cm_xs = CovMat(num_groups=2, energy_grid=[1e-5, 1.0, 20e6])
cm_xs.add_matrix(1001, 18, 1001, 18, np.eye(2) * 0.01)
cm_xs.cross_sections[(1001, 18)] = np.array([5.0, 10.0])

# 13a: COVERX binary emits float32 warning
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    write_coverx(cm_xs, os.path.join(tmpdir, 'warn_f32.bin'), fmt='binary')
    f32_warns = [x for x in w if 'float32' in str(x.message)]
    assert len(f32_warns) >= 1, 'Expected float32 warning from COVERX binary writer'
    print(f'COVERX binary float32: {f32_warns[0].message}')

# 13b: COVERX text emits cross-section loss warning
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    write_coverx(cm_xs, os.path.join(tmpdir, 'warn_xs.txt'), fmt='text')
    xs_warns = [x for x in w if 'cross-section storage' in str(x.message)]
    assert len(xs_warns) >= 1, 'Expected XS loss warning from COVERX text writer'
    print(f'COVERX text XS loss:  {xs_warns[0].message}')

# 13c: COVERX text should NOT warn if no cross-sections
cm_no_xs = CovMat(num_groups=2, energy_grid=[1e-5, 1.0, 20e6])
cm_no_xs.add_matrix(1001, 18, 1001, 18, np.eye(2) * 0.01)
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    write_coverx(cm_no_xs, os.path.join(tmpdir, 'no_warn_xs.txt'), fmt='text')
    xs_warns = [x for x in w if 'cross-section storage' in str(x.message)]
    assert len(xs_warns) == 0, f'Unexpected XS loss warning when no XS present: {xs_warns}'
    print('No false-positive XS loss warning: PASS')

print('PASS: COVERX format warnings working correctly')

## Test 14: BOXER low-precision warning

NVF <= 10 should emit a warning suggesting higher precision.

In [ ]:
cm_prec = CovMat(num_groups=2, energy_grid=[1e-5, 1.0, 20e6])
cm_prec.add_matrix(1001, 18, 1001, 18, np.eye(2) * 0.01)

# 14a: NVF=10 (default) should warn
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    write_boxer(cm_prec, os.path.join(tmpdir, 'prec_nvf10.boxer'), nvf=10)
    prec_warns = [x for x in w if 'NVF=' in str(x.message)]
    assert len(prec_warns) >= 1, 'Expected precision warning for NVF=10'
    print(f'NVF=10 warning: {prec_warns[0].message}')

# 14b: NVF=14 should NOT warn about precision
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    write_boxer(cm_prec, os.path.join(tmpdir, 'prec_nvf14.boxer'), nvf=14)
    prec_warns = [x for x in w if 'NVF=' in str(x.message)]
    assert len(prec_warns) == 0, f'Unexpected precision warning for NVF=14: {prec_warns}'
    print('No precision warning for NVF=14: PASS')

# 14c: NVF=7 should also warn (even lower precision)
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    write_boxer(cm_prec, os.path.join(tmpdir, 'prec_nvf7.boxer'), nvf=7)
    prec_warns = [x for x in w if 'NVF=' in str(x.message)]
    assert len(prec_warns) >= 1, 'Expected precision warning for NVF=7'
    print(f'NVF=7 warning:  {prec_warns[0].message}')

print('PASS: BOXER precision warnings working correctly')

## Test 15: MeV-to-eV energy conversion

`write_covfil` and `write_boxer` must convert MeV grids to eV and emit a warning.
The written files should contain eV values that match the original MeV * 1e6.

In [ ]:
egrid_mev = [1e-11, 1e-6, 20.0]
egrid_ev = [e * 1e6 for e in egrid_mev]

cm_mev = CovMat(num_groups=2, energy_grid=egrid_mev, energy_unit='MeV')
cm_mev.add_matrix(1001, 18, 1001, 18, np.eye(2) * 0.01)

# 15a: COVFIL MeV->eV conversion
out_covfil_mev = os.path.join(tmpdir, 'mev_to_ev.covfil')
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    write_covfil(cm_mev, out_covfil_mev)
    mev_warns = [x for x in w if 'MeV to eV' in str(x.message)]
    assert len(mev_warns) >= 1, 'Expected MeV->eV warning from write_covfil'
    print(f'COVFIL MeV->eV: {mev_warns[0].message}')

# Read back and check energy values are in eV
cm_covfil_rt = read_covfil(out_covfil_mev)
assert np.allclose(cm_covfil_rt.energy_grid, egrid_ev, rtol=1e-6), \
    f'COVFIL energy grid should be in eV: {cm_covfil_rt.energy_grid}'
print(f'COVFIL read-back grid (eV): {cm_covfil_rt.energy_grid}')

# 15b: BOXER MeV->eV conversion
out_boxer_mev = os.path.join(tmpdir, 'mev_to_ev.boxer')
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    write_boxer(cm_mev, out_boxer_mev, nvf=14)
    mev_warns = [x for x in w if 'MeV to eV' in str(x.message)]
    assert len(mev_warns) >= 1, 'Expected MeV->eV warning from write_boxer'
    print(f'BOXER MeV->eV:  {mev_warns[0].message}')

# Read back and check energy values are in eV
cm_boxer_rt = read_boxer(out_boxer_mev)
assert np.allclose(cm_boxer_rt.energy_grid, egrid_ev, rtol=1e-5), \
    f'BOXER energy grid should be in eV: {cm_boxer_rt.energy_grid}'
print(f'BOXER read-back grid (eV): {cm_boxer_rt.energy_grid}')

# 15c: No warning when already in eV
cm_ev = CovMat(num_groups=2, energy_grid=[1e-5, 1.0, 20e6], energy_unit='eV')
cm_ev.add_matrix(1001, 18, 1001, 18, np.eye(2) * 0.01)
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    write_covfil(cm_ev, os.path.join(tmpdir, 'ev_noop.covfil'))
    mev_warns = [x for x in w if 'MeV to eV' in str(x.message)]
    assert len(mev_warns) == 0, f'Unexpected MeV warning for eV data: {mev_warns}'
    print('No false-positive MeV->eV warning for eV data: PASS')

print('PASS: MeV->eV conversion working correctly in all writers')

In [ ]:
print('\n=== All cross-format tests completed ===')
print(f'Temp files in: {tmpdir}')